# OWL-style materialization with runtime N3 rules

pyling can load rule profiles at runtime. This small notebook uses an embedded OWL-style subset to demonstrate subclass, domain, range, inverse property, and symmetric property materialization. Larger OWL2RL runs use the benchmark harness and the external RDFJS/Eyeling rule file documented in the README.

In [1]:
from pyling import reason_stream

rules = """
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
{ ?x rdf:type ?class . ?class rdfs:subClassOf ?superClass . } => { ?x rdf:type ?superClass } .
{ ?p rdfs:domain ?class . ?x ?p ?y . } => { ?x rdf:type ?class } .
{ ?p rdfs:range ?class . ?x ?p ?y . } => { ?y rdf:type ?class } .
{ ?p owl:inverseOf ?q . ?x ?p ?y . } => { ?y ?q ?x } .
{ ?p rdf:type owl:SymmetricProperty . ?x ?p ?y . } => { ?y ?p ?x } .
"""

data = """
@prefix : <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
:authored rdfs:domain :Researcher ; rdfs:range :Paper ; owl:inverseOf :hasAuthor .
:collaboratesWith rdf:type owl:SymmetricProperty .
:Researcher rdfs:subClassOf :Person .
:alice :authored :paper42 ; :collaboratesWith :bob .
"""

result = reason_stream({"sources": [rules, data]}, include_input_facts_in_closure=True)
print(result.closure_n3)

@prefix : <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:Researcher rdfs:subClassOf :Person .
:alice :authored :paper42 .
:alice :collaboratesWith :bob .
:alice a :Person .
:alice a :Researcher .
:authored owl:inverseOf :hasAuthor .
:authored rdfs:domain :Researcher .
:authored rdfs:range :Paper .
:bob :collaboratesWith :alice .
:collaboratesWith a owl:SymmetricProperty .
:paper42 :hasAuthor :alice .
:paper42 a :Paper .



In [2]:
expected = [
    ":alice a :Researcher .",
    ":alice a :Person .",
    ":paper42 a :Paper .",
    ":paper42 :hasAuthor :alice .",
    ":bob :collaboratesWith :alice .",
]
for triple in expected:
    print(triple, triple in result.closure_n3)

:alice a :Researcher . True
:alice a :Person . True
:paper42 a :Paper . True
:paper42 :hasAuthor :alice . True
:bob :collaboratesWith :alice . True
